## 1_logreg_fairness.ipynb ##
## Logistic Regression with Fairness Constraints
### Fairness and Compliance of AI
### University of Oklahoma

__Mitigating Disparate Impact using Zafar's Constraints__

This notebook demonstrates how to improve fairness metric scores by incorporating fairness constraints directly into a model's optimization process. We use the UCI Adult Census Income dataset to predict whether an individual earns more than $50,000/year, focusing on Sex as the protected (sensitive) attribute.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.special import expit
import random

random.seed(42)

# --- 1. Load the Dataset ---
try:
    columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", 
            "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", 
            "hours-per-week", "native-country", "income"]

    data = pd.read_csv("adult.data", names=columns, sep=r',\s*', engine='python')
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)


# --- 2. Feature Engineering & Simplification ---

# Simplify Workclass (Target: Max 5 categories)

# Replace the '?' with NaN values
data.replace('?', np.nan, inplace=True)

work_map = {
    'Private': 'Private',
    'Self-emp-not-inc': 'Self-Employed', 'Self-emp-inc': 'Self-Employed',
    'Local-gov': 'Government', 'State-gov': 'Government', 'Federal-gov': 'Government',
    'Without-pay': 'Other', 'Never-worked': 'Other', '?': 'Other'
}
data['work_simple'] = data['workclass'].map(work_map)

# Simplify Marital Status (Target: 3 categories)
mar_map = {
    'Married-civ-spouse': 'Married', 'Married-AF-spouse': 'Married', 'Married-spouse-absent': 'Married',
    'Divorced': 'Divorced', 'Separated': 'Divorced', 'Widowed': 'Divorced',
    'Never-married': 'Never-married'
}
data['mar_simple'] = data['marital-status'].map(mar_map)

# Targets and Sensitive Attributes
data['target'] = data['income'].apply(lambda x: 1 if '>50K' in x else 0)
data['sex_binary'] = data['sex'].apply(lambda x: 1 if x == 'Male' else 0)

# Collapse Amer-Indian-Eskimo and Other into a single category "Other" for simplicity
data['race'] = data['race'].replace({'Amer-Indian-Eskimo': 'Other', 'Other': 'Other'})

# --- 3. Define Predictor Groups ---
# Continuous: We WILL scale these
continuous_features = ['age', 'education-num', 'capital-gain', 'hours-per-week']

# Categorical/Binary: We WILL NOT scale these
categorical_features = ['work_simple', 'mar_simple', 'race']
# 'sex_binary' is also categorical, but we handle it as our sensitive attribute

# Look at the data after simplification
print(data.head())

### Transform variables
One-hot encode categorical variables and z-score normalize continuous ones

In [ ]:
# --- 4. Transform variables ---

# Create One-Hot Encodings for categories
# drop_first=True prevents the 'dummy variable trap' (multi-collinearity)
X_cat = pd.get_dummies(data[categorical_features], drop_first=True)

# Extract Numeric features
X_num = data[continuous_features]

# Create the full feature set (Adding sex_binary as a predictor as requested)
X_combined = pd.concat([X_num, X_cat, data[['sex_binary']]], axis=1).astype(float)
y = data['target']
A = data['sex_binary']

# Split the data BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X_combined, y, A, test_size=0.3, random_state=42
)

# SELECTIVE SCALING
# We only want to scale the columns that were originally continuous
scaler = StandardScaler()

# Scaling Training Data
X_train_scaled = X_train.copy()
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])

# Scaling Testing Data
X_test_scaled = X_test.copy()
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Note: All binary columns (Work, Marital, Race, Sex) remain exactly 0 or 1.

### Setup a set of fairness metrics
The metrics we include here are Demographic Parity, Equal Odds (Error-Rate Parity), and Predictive Parity

In [ ]:
# --- 5. Define fairness metrics ---

def check_fairness(y_true, y_pred, sensitive_attr):
    """
    Evaluates a model's fairness by comparing performance across different groups.
    
    Parameters:
    y_true : Series/Array - The actual ground truth labels (0 or 1)
    y_pred : Series/Array - The model's binary predictions (0 or 1)
    sensitive_attr : Series/Array - The binary sensitive feature (e.g., Sex: 1=Male, 0=Female)
    """
    
    # Create a temporary dataframe to align predictions with actuals and sensitive labels
    results = pd.DataFrame({'y': y_true, 'y_hat': y_pred, 'A': sensitive_attr})
    
    # Split the results into two groups based on the sensitive attribute
    # Group 0 is often the historically 'unprivileged' or 'protected' group
    g0 = results[results['A'] == 0] 
    g1 = results[results['A'] == 1] 
    
    # Dictionary to store calculated metrics for each group
    stats = {}
    
    for name, group in [('Female', g0), ('Male', g1)]:
        # Calculate the Confusion Matrix components:
        # TN: True Negatives, FP: False Positives, FN: False Negatives, TP: True Positives
        # labels=[0,1] ensures the matrix is always 2x2 even if a group is missing a class
        tn, fp, fn, tp = confusion_matrix(group['y'], group['y_hat'], labels=[0, 1]).ravel()
        
        # 1. Demographic Parity (Statistical Independence)
        # Ratio of positive predictions regardless of the actual ground truth.
        # Ideally, P(Y_hat=1 | A=0) == P(Y_hat=1 | A=1)
        dp = group['y_hat'].mean()
        
        # 2. True Positive Rate (TPR / Sensitivity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'High Income' for those who actually have it?
        # A gap here indicates the model 'misses' qualified candidates in one group more than the other.
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # 3. True Negative Rate (TNR / Specificity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'Low Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        # 4. Predictive Parity (Sufficiency)
        # If the model predicts 'High Income', how likely is it to be correct?
        # This checks if the 'meaning' of a positive prediction is the same across groups.
        pp = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Store metrics for this group
        stats[name] = {'DP': dp, 'TPR': tpr, 'TNR': tnr, 'PP': pp}

    # --- Print Results Table ---
    print(f"{'Metric':<22} | {'Female':<8} | {'Male':<8} | {'Difference':<10}")
    print("-" * 65)
    
    # Iterate through the metrics to calculate and display the absolute difference
    for m in ['DP', 'TPR', 'TNR', 'PP']:
        # Rename keys for the final display
        display_names = {
            'DP': 'Demographic Parity',
            'TPR': 'True Positive Rate',
            'TNR': 'True Negative Rate',
            'PP': 'Predictive Parity'
        }
        
        f_val = stats['Female'][m]
        m_val = stats['Male'][m]
        diff = abs(f_val - m_val)
        
        print(f"{display_names[m]:<22} | {f_val:.3f}   | {m_val:.3f}   | {diff:.3f}")
        
    print("-" * 65)
    print("Note: 'Error Rate Parity' is satisfied if both TPR and TNR differences are near zero.")


### Run the unconstrained logistic regression

In [ ]:
# --- 6. Run baseline model ---
#Train Baseline
baseline = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred_base = baseline.predict(X_test_scaled)
y_prob_base = baseline.predict_proba(X_test_scaled)[:, 1]

print("=== BASELINE RESULTS (Unconstrained) ===")
check_fairness(y_test, y_pred_base, A_test)

### Run the LR with Fairness Constraints

In [ ]:
# --- 7. Run model with fairness constraints ---

# Objective: Minimize standard Log Loss (same as cross-entropy)

def objective(theta, X, y, alpha=1e-4):
    X_ext = np.hstack([np.ones((X.shape[0], 1)), X])
    z = X_ext.dot(theta)
    # stable logistic loss: log(1+exp(z)) - y*z
    loss = np.sum(np.logaddexp(0, z) - y * z)
    # small L2 regularization (exclude intercept)
    loss += 0.5 * alpha * np.sum(theta[1:]**2)
    return loss


# Constraint: Covariance between sensitive attribute and decision boundary < c
def fairness_constraint(theta, X, A, c):
    X_ext = np.hstack([np.ones((X.shape[0], 1)), X])
    distances = np.dot(X_ext, theta)
    # The 'unfairness' is measured by how much 'A' correlates with predicted distance
    cov = np.mean((A - np.mean(A)) * distances)
    return c - np.abs(cov)

# Run Optimization
c_val = 0.001 
initial_weights = np.zeros(X_train_scaled.shape[1] + 1)

res = minimize(
    fun=objective, x0=initial_weights, args=(X_train_scaled, y_train),
    constraints={'type': 'ineq', 'fun': fairness_constraint, 'args': (X_train_scaled, A_train, c_val)},
    method='SLSQP', options={'maxiter': 500},
    )

# Fair Predictions (use numerically-stable sigmoid)
X_test_ext = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled])
fair_prob = expit(np.dot(X_test_ext, res.x))
fair_pred = (fair_prob > 0.5).astype(int)

print("\n=== FAIR MODEL RESULTS (Constrained) ===")
check_fairness(y_test, fair_pred, A_test)

### Compare the results

In [ ]:
# --- 8. Visualize Results ---

fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_base)
fpr_f, tpr_f, _ = roc_curve(y_test, fair_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr_b, tpr_b, label=f"Baseline (AUC={auc(fpr_b, tpr_b):.3f})", color='gray', linestyle='--')
plt.plot(fpr_f, tpr_f, label=f"Fair Model (AUC={auc(fpr_f, tpr_f):.3f})", color='blue', linewidth=2)
plt.plot([0, 1], [0, 1], 'k:', alpha=0.5)
plt.title("ROC Curve Comparison: Baseline vs Fairness-Constrained Model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

### Notes on Elements of the Code

__The Objective Function: Cross-Entropy and Numerical Stability__
The cell that defines the objective function and constraints for the fairness constrained method has a few tweaks to it to avoid computational issues.  This markdown summarizes what's going on in that code block. You are not required to understand this for the class, but you may find it helpful in your own work.

The objective function is the minimization of the loss function, which is cross-entropy loss as described in class. However, the form of the equation differs slightly through some simplification. Also, we need to add two enhancements:
+ Numerical Stability: we modify the function to prevent the explosion of $\sigma$
+ $L_2$ Regularization: we seek to limit the size of the regression coefficients

In class, we defined the Binary Cross-Entropy loss for a single data point $i$ as:
$$L_i = -(y_i \log(\hat{y_i}) + (1 - y_i) \log(1 - \hat{y_i}))$$
In Logistic Regression, we define the predicted outcome, which is a probability, $\hat{y_i}$ using the sigmoid function $\sigma(z_i)$ of our linear predictor $z_i$ (where $z = W^T\mathbf{x}+\mathbf{b}$):
$$\hat{y_i} = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}$$
When we substitute the sigmoid function into the Cross-Entropy formula and simplify the algebra, we arrive at the Logistic Loss formula used in our code:$$L_i = \log(1 + e^{z_i}) - y_i z_i$$
__Numerical Stability via logaddexp:__ Calculating $\log(1 + e^z)$ directly is computationally "dangerous." If the model becomes very confident and $z$ becomes a large number (e.g., $z > 700$), the term $e^z$ will overflow to infinity. This results in a NaN (Not a Number) error, which crashes the optimization process. To prevent this, we use the Log-Sum-Exp trick via the NumPy function np.logaddexp(0, z). Mathematically, $\log(e^a + e^b)$ is calculated. By setting $a=0$, we get $\log(e^0 + e^z)$, which is exactly $\log(1 + e^z)$. Internally, logaddexp re-writes the calculation to avoid large exponents:$$\text{logaddexp}(0, z) = \max(0, z) + \log(1 + e^{-|z|})$$
Since the exponent is now always negative or zero ($e^{-|z|}$), the value is always between 0 and 1. This prevents the "Infinity" crash and allows the solver to handle large weights safely.

$\mathbf{L_2}$ __Regularization Penalty:__ A basic loss function only cares about fitting the training data perfectly. Our Objective Function adds a second component called a Penalty Term (also known as Ridge Regularization):$$\text{Penalty} = \frac{1}{2} \alpha \sum_{j=1}^{p} \theta_j^2$$
Why add this? It prevents any single feature from having an "explosive" or excessively large weight. This "smoothes" the optimization surface, helping the solver find a stable path toward a fair solution. The Intercept Rule: Notice the code uses theta[1:]. We do not regularize the intercept ($\theta_0$). The intercept shifts the decision boundary to center it on the data; penalizing it would bias the model's "default" prediction regardless of the evidence.Summary: The Total ObjectiveThe final function we minimize is the sum of the stable losses plus the penalty:$$J(\theta) = \underbrace{\sum_{i=1}^{n} \left[ \text{logaddexp}(0, z_i) - y_i z_i \right]}_{\text{Total Cross-Entropy (Accuracy)}} + \underbrace{\frac{\alpha}{2} \sum_{j=1}^{p} \theta_j^2}_{\text{Regularization (Complexity)}}$$By minimizing this objective subject to Zafar's Fairness Constraints, we find the most accurate possible model that remains within our allowed threshold for disparate impact.